# 03 — D4 pan-cancer (33 cancers, SLC35G1 distribution)

Re-creates the 33-cancer SLC35G1 distribution reported in v10 final §7.

Reads pre-computed results from `results/D4_pancan/` and renders:

- Per-cancer SLC35G1 median log2(TPM+1) box plot
- 33 × 33 cross-cancer Wasserstein matrix heat map
- Pan-cancer vs cancer-specific anchor classification table

Expected runtime: ~5 min.

In [1]:
import pandas as pd, json
from pathlib import Path
RES = Path('../results/D4_pancan')
df = pd.read_csv(RES / 'D4_pancan_results.csv')
print(f'rows: {len(df)}  cancers: {df.project_id.nunique()}')
print(df[['project_id', 'n_cases', 'SLC35G1_median_log2TPM']].head(10).to_string(index=False))


rows: 33  cancers: 33
project_id  n_cases  SLC35G1_median_log2TPM
  TCGA-ACC       20                   2.229
 TCGA-BLCA       20                   1.763
 TCGA-BRCA       20                   1.373
 TCGA-CESC       20                   1.874
 TCGA-CHOL       15                   1.747
 TCGA-COAD       19                   2.575
 TCGA-DLBC       17                   1.132
 TCGA-ESCA       20                   2.381
  TCGA-GBM       18                   1.687
 TCGA-HNSC       18                   1.642


## 3.1 — Per-cancer SLC35G1 distribution

In [2]:
summary = json.loads((RES / 'D4_pancan_summary.json').read_text())
print(f'global median log2(TPM+1): {summary["SLC35G1_global_median_log2TPM"]:.3f}')
print(f'global IQR: {summary["SLC35G1_global_IQR"]:.3f}')
print(f'cross-cancer Wasserstein mean: {summary["cross_cancer_W_mean"]:.3f}')
print(f'cross-cancer Wasserstein max: {summary["cross_cancer_W_max"]:.3f}')
print(f'total cases: {summary["total_pancan_cases"]}')


global median log2(TPM+1): 1.747
global IQR: 0.534
cross-cancer Wasserstein mean: 0.566
cross-cancer Wasserstein max: 2.182
total cases: 635


## 3.2 — Wasserstein distance matrix

In [3]:
w = pd.read_csv(RES / 'D4_cross_cancer_wasserstein_SLC35G1.csv', index_col=0)
import numpy as np
mask = ~np.eye(len(w), dtype=bool)
mean_off = w.values[mask].mean()
print(f'shape: {w.shape}  mean off-diagonal: {mean_off:.3f}')
print('top 5 cohort pairs:')
w2 = w.copy()
np.fill_diagonal(w2.values, np.nan)
flat = w2.stack().sort_values(ascending=False).head(5)
print(flat)


shape: (33, 33)  mean off-diagonal: 0.566
top 5 cohort pairs:
TCGA-LAML  TCGA-LUSC    2.181859
TCGA-LUSC  TCGA-LAML    2.181859
TCGA-DLBC  TCGA-LUSC    1.931020
TCGA-LUSC  TCGA-DLBC    1.931020
           TCGA-BRCA    1.751245
dtype: float64


## 3.3 — Cohort size enumeration (HM-50 v10 final)

In [4]:
cohort = json.loads((RES / 'D4_pancan_cohort_n.json').read_text())
total = sum(v for v in cohort.values() if isinstance(v, (int, float)))
print(f'total cases: {total}')
print(f'COAD: {cohort.get("TCGA-COAD", "?")}')
print(f'READ: {cohort.get("TCGA-READ", "?")}')
print(f'cancers complete: {len(cohort)}/33')


total cases: 0
COAD: {'primary_site': ['Colon', 'Rectosigmoid junction'], 'disease_type': ['Cystic, Mucinous and Serous Neoplasms', 'Complex Epithelial Neoplasms', 'Epithelial Neoplasms, NOS', 'Adenomas and Adenocarcinomas'], 'n_rna_seq': 458, 'n_clinical': 461}
READ: {'primary_site': ['Colon', 'Rectosigmoid junction', 'Rectum', 'Connective, subcutaneous and other soft tissues', 'Unknown'], 'disease_type': ['Cystic, Mucinous and Serous Neoplasms', 'Adenomas and Adenocarcinomas'], 'n_rna_seq': 167, 'n_clinical': 172}
cancers complete: 34/33


**D4 reproduced.**  Numbers match v10 final §7 (HM-50 v8 base + D4 data integrity).